# LILA ALIA Augmentation Notebook

Use the class list serialized from `lila-suppl-data.csv`, sample images from `lila_splits/lila_splits_train.csv`, and generate augmentations with the ALIA pipeline.

This notebook keeps the sampling rule deterministic and saves the manifests needed for later SpeciesNet fine-tuning.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

from lila_pipeline_helpers import (
    load_lila_suppl_data,
    select_bottom_percent_classes,
    build_class_targets,
    serialize_class_targets,
    load_lila_train_split,
    sample_class_images,
    write_manifest,
)

ROOT = Path.cwd()
ARTIFACT_DIR = ROOT / 'artifacts' / 'lila_alia_augmentation'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ALIA_REPO = ROOT / 'old_alia' / 'ALIA'


In [ ]:
from pathlib import Path
import sys
import pandas as pd

from lila_pipeline_helpers import (
    load_lila_suppl_data,
    select_bottom_percent_classes,
    build_class_targets,
    serialize_class_targets,
    load_lila_train_split,
    sample_class_images,
    write_manifest,
)

ROOT = Path.cwd()
ARTIFACT_DIR = ROOT / 'artifacts' / 'lila_alia_augmentation'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ALIA_REPO = ROOT / 'old_alia' / 'ALIA'
sys.path.insert(0, str(ALIA_REPO))
from alia_speciesnet_helpers import build_alia_augmentations_from_repo, ALIA_EDIT_SPECS


In [ ]:
train_df = load_lila_train_split(ROOT / 'lila_splits' / 'lila_splits_train.csv')
train_df = train_df[train_df['common_name'].isin(targets['common_name'])].reset_index(drop=True)

# Build the exact rows that will be augmented.
selection_rows = []
for _, row in targets.iterrows():
    class_name = row['common_name']
    n_new = int(row['target_added'])
    if n_new <= 0:
        continue
    sample_n = min(n_new, len(train_df[train_df['common_name'] == class_name]))
    sampled = sample_class_images(train_df, class_name, n=sample_n, seed=42)
    sampled = sampled.assign(target_added=n_new, class_target=class_name)
    selection_rows.append(sampled)

selection_df = pd.concat(selection_rows, ignore_index=True) if selection_rows else pd.DataFrame(columns=train_df.columns)
write_manifest(selection_df, ARTIFACT_DIR / 'alia_source_selection.csv')
selection_df.head()


In [ ]:
# Sample source rows from the underrepresented classes and run the ALIA repo pipeline.
# The source rows use gs_filepath from lila_splits/lila_splits_train.csv.
source_rows = []
for _, row in targets.iterrows():
    class_name = row['common_name']
    n_new = int(row['target_added'])
    if n_new <= 0:
        continue
    class_df = train_df[train_df['common_name'] == class_name].reset_index(drop=True)
    if class_df.empty:
        continue
    # Reuse real train images; generation happens in the ALIA repo downstream.
    sampled = sample_class_images(class_df, class_name, n=min(n_new, len(class_df)), seed=42)
    source_rows.append(sampled.assign(class_target=class_name, target_added=n_new))

source_df = pd.concat(source_rows, ignore_index=True) if source_rows else pd.DataFrame(columns=train_df.columns)
source_df['path'] = source_df['gs_filepath']
write_manifest(source_df, ARTIFACT_DIR / 'alia_source_selection.csv')
source_df.head()


In [ ]:
# Run ALIA's image generation on the selected source rows.
# This follows the upstream ALIA flow: source image -> prompt -> img2img edit.
# If you want to change the prompt family, edit methods/prompt_strategy below.
augmented = build_alia_augmentations_from_repo(
    source_df=source_df,
    output_root=ARTIFACT_DIR / 'generated_images',
    repo_path=ALIA_REPO,
    max_images_per_class=999999,
    methods=[s for s in ALIA_EDIT_SPECS if s.name in {'background', 'weather', 'lighting', 'season'}],
    prompt_strategy='prompt_bank',
    species_lookup=None,
    n=1,
)
write_manifest(augmented, ARTIFACT_DIR / 'alia_generated_manifest.csv')
augmented.head()


## Next step
Run the ALIA repo workflow using the serialized class list and source-selection manifest, then feed the generated images into the frozen-backbone SpeciesNet fine-tuning pass.

## SpeciesNet fine-tuning
Freeze the backbone and train only the classifier head on the added images.

In [ ]:
from lila_pipeline_helpers import freeze_backbone, trainable_parameter_names

# model = load_speciesnet_model(...)
# freeze_backbone(model)
# optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
# print(trainable_parameter_names(model))
